# 3D workspace visualization for the Kinematics repository.

### What it shows:

- Reachable gripper-center positions based on configured link lengths and joint limits
- The configured workspace box from configs/kinematics_settings.toml
- One arm pose with shoulder, elbow, wrist, and gripper-center points

### Target pose (optional):

```python tools/visualize_workspace.py --target 250 850 0```

- X = left/right in robot-base coordinates
- Z = front/back in robot-base coordinates
- Y = distance downward from the top reference

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from config.config_loader import load_config

ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = ROOT / "src"

def joint_by_role(servo_config: dict[str, Any], role: str) -> dict[str, Any]:
    """Return the servo joint config matching a kinematic role such as theta1."""
    for joint in servo_config["joints"].values():
        if joint.get("kinematic_role") == role:
            return joint

    raise KeyError(f"No joint with kinematic_role={role!r} found.")

In [ ]:
@dataclass(frozen=True)
class RobotConfig:
    """Relevant values loaded from the TOME configuration file."""
    repo_root: Path
    geometry: dict[str, Any]
    settings: dict[str, Any]
    servo: dict[str, Any]

    l1_mm: float
    l2_mm: float
    lg_mm: float
    h0_mm: float

    base_x_mm: float
    base_y_mount_mm: float
    base_z_mm: float
    shoulder_y_mm: float

    approach_angle_deg: float
    bounds: dict[str, float]

    j1_min_deg: float
    j1_max_deg: float
    j2_min_deg: float
    j2_max_deg: float
    j3_min_deg: float
    j3_max_deg: float

    solution_preference: str
    check_side_view_orientation: bool

    @staticmethod
    def from_files(repo_root: Path, config_dir: Path) -> "RobotConfig":
        geometry = load_config("robot_geometry.toml")
        settings = load_config("kinematics_settings.toml")
        servo = load_config("servo_calibration.toml")

        model = settings["model"]
        links = geometry["link_lengths_mm"]

        l1 = float(links["L1_shoulder_to_elbow"])
        l2 = float(links["L2_elbow_to_wrist"])

        if model["use_gripper_offset"]:
            lg = float(links[model["selected_Lg_key"]])
        else:
            lg = 0.0

        if model["use_h0_from_robot_geometry"]:
            h0 = float(links[model["selected_h0_key"]])
        else:
            h0 = 0.0

        input_coordinates = settings.get("input_coordinates", {})
        base_origin = input_coordinates.get(
            "base_rotation_axis_at_mounting_plate_mm",
            [0.0, input_coordinates.get("max_height_mm", 0.0), 0.0]
        )

        base_x, base_y_mount, base_z = [float(value) for value in base_origin]
        shoulder_y = base_y_mount - h0

        j1 = joint_by_role(servo, "theta1")
        j2 = joint_by_role(servo, "theta2")
        j3 = joint_by_role(servo, "theta3")

        validation = settings["validation"]

        return RobotConfig(
            repo_root=repo_root,
            geometry=geometry,
            settings=settings,
            servo=servo,
            l1_mm=l1,
            l2_mm=l2,
            lg_mm=lg,
            h0_mm=h0,
            base_x_mm=base_x,
            base_y_mount_mm=base_y_mount,
            base_z_mm=base_z,
            shoulder_y_mm=shoulder_y,
            approach_angle_deg=float(settings["ik"]["default_approach_angle_deg"]),
            bounds={key: float(value) for key, value in settings["workspace_bounds_robot_base_mm"].items()},
            j1_min_deg=float(j1["theta_min_deg"]),
            j1_max_deg=float(j1["theta_max_deg"]),
            j2_min_deg=float(j2["theta_min_deg"]),
            j2_max_deg=float(j2["theta_max_deg"]),
            j3_min_deg=float(j3["theta_min_deg"]),
            j3_max_deg=float(j3["theta_max_deg"]),
            solution_preference=str(settings["ik"]["solution_preference"]),
            check_side_view_orientation=bool(validation.get("check_side_view_orientation", False)),
        )


